# 06 — Power BI: diez dashboards y modelo semántico

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Inventario de páginas y visuales

El PBIP contiene 3 páginas descriptivas, 3 diagnósticas, 3 predictivas y 1 de auditoría.
Los contratos provienen de Gold y auditoría; las medidas se calculan en el modelo semántico.

**QUÉ EXPLICAR:** cada página responde una pregunta de decisión, no es solo una colección
de gráficos.

In [ ]:
pages_root = ROOT / 'powerbi' / 'TLC_BigData.Report' / 'definition' / 'pages'
order = json.loads((pages_root / 'pages.json').read_text(encoding='utf-8'))['pageOrder']
inventory = []
for page_id in order:
    page_dir = pages_root / page_id
    page = json.loads((page_dir / 'page.json').read_text(encoding='utf-8'))
    visuals = list((page_dir / 'visuals').glob('*/visual.json'))
    inventory.append((page['displayName'], len(visuals)))
for item in inventory:
    print(item)
assert len(inventory) == 10

### 2. Hechos, dimensiones y relaciones

In [ ]:
semantic = ROOT / 'powerbi' / 'TLC_BigData.SemanticModel' / 'definition'
table_files = list((semantic / 'tables').glob('*.tmdl'))
facts = [p.stem for p in table_files if p.stem.startswith('Fact_')]
dims = [p.stem for p in table_files if p.stem.startswith('Dim')]
relation_text = (semantic / 'relationships.tmdl').read_text(encoding='utf-8')
relationships = relation_text.count('relationship ')
print('Hechos:', len(facts), facts)
print('Dimensiones:', len(dims), dims)
print('Relaciones:', relationships)

## Comprobaciones

In [ ]:
report = json.loads((EXPORTS / 'verification_report.json').read_text(encoding='utf-8'))
check = next(c for c in report['checks'] if c['name'] == 'powerbi_ten_pages')
definition_files = [
    p for p in (ROOT / 'powerbi').rglob('*')
    if p.is_file() and p.suffix.lower() in {'.json', '.tmdl', '.pbip', '.pbir', '.pbism'}
]
all_text = '\n'.join(
    p.read_text(encoding='utf-8', errors='ignore') for p in definition_files
)
assert check['passed'] and check['details']['pages'] == 10
assert 'PLACEHOLDER' not in all_text
assert len(facts) == 15 and len(dims) == 7
print('Power BI validado:', check['details']['visuals'])

## Siguiente paso

Abra `powerbi/TLC_BigData.pbip`, actualice y recorra las páginas 01–10. Cierre mostrando
la página 10 y el reporte de verificación sin fallos.